# Did the world model anticipate the raw-MSE detour?

Use a GPU runtime. This audit starts with the green upper-goal rollout, **b1_g04_s01**, from the earlier detour figure. The original run stopped at action 23. Its episode record saves all five commanded macro-actions, including the final two commands it never executed.

We reconstruct the world model's predictions from the recorded start and commands and compare with the actual ball position and true latent MSE at the complete macro boundaries. A separate simulator replay continues to action 25 and is only included if it reproduces the recorded prefix. CEM is rerun once to check reproducibility. The saved run is read-only.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import io
import json
import numpy as np
import pandas as pd
import torch

drive.mount('/content/drive')
RUNS=Path('/content/drive/MyDrive/datasets/curriculum_cost_adapter/experience_temporal_half_square_v1')
RUN_ID='20261107T150425413490Z'  # The run used in the attached audit.
CASE_ID='b1_g04_s01'
VERIFY_CEM=True
SUCCESS_RADIUS=4.5

## Read the archived commands and observations

The archived macro-actions must exactly match the executed primitive prefix. The diagnostic requires a rollout produced by one planner call.

In [ ]:
def read_json(path):
    return json.loads(path.read_text())

run=RUNS/RUN_ID
stage=read_json(run/'stages/band_1.json')
assert stage['status']=='complete' and stage['condition']['collector']=='raw'
manifest=read_json(run/'experiment.json')['manifest']
cases=manifest['stages'][0]
assert stage['cases_digest']==hashlib.sha256(json.dumps(cases,sort_keys=True,allow_nan=False).encode()).hexdigest()
case=next(c for c in cases if c['case_id']==CASE_ID)
record=read_json(run/f'episodes/band_1/{CASE_ID}.json')
assert record['case']==case and record['condition']==stage['condition'] and record['runtime_id']==stage['runtime_id']
assert record['mpc_updates']==1, 'Select a band 1 episode with exactly one planner call.'
raw_path=(run/record['raw_path']).resolve()
raw_path.relative_to(run.resolve())
data=raw_path.read_bytes()
assert hashlib.sha256(data).hexdigest()==record['raw_sha256'], 'Raw rollout checksum differs.'
raw=torch.load(io.BytesIO(data),map_location='cpu',weights_only=True)
del data
assert raw['case']==case and raw['runtime_id']==stage['runtime_id'] and raw['success']==record['success']
steps=record['micro_steps']
actual_xy=raw['states'][:,:2].numpy()
assert len(actual_xy)==len(raw['frames'])==steps+1
assert np.array_equal(raw['primitive_steps'].numpy(),np.arange(steps+1))
assert np.array_equal(raw['primitive_actions'].numpy(),np.asarray(record['primitive_actions'],dtype=np.float32))
planning=manifest['planning']
print('Case:',CASE_ID,'· start:',case['start'],'· goal:',case['goal'])
print('Recorded simulator actions:',steps,'· macro durations:',record['macro_durations'])

## Restore the original models and planning settings

Both model files are verified against the collection runtime, and the original goal tensor is reused. For an exact CEM reproduction, use the same GPU model and PyTorch build shown below. The archived actions remain the reference if CEM returns a different sequence.

In [ ]:
import re
import shutil
import subprocess
import sys
from types import SimpleNamespace

assert torch.cuda.is_available(), 'Select a GPU runtime.'
runtime=read_json(run/'runtime.json')
signature=runtime['signature']
stage=read_json(run/'stages/band_1.json')
assert runtime['runtime_id']==stage['runtime_id'], 'Band 1 runtime changed.'

def checkout(url,path,commit,branch=None):
    assert re.fullmatch(r'[a-f0-9]{40}',commit), 'Invalid recorded commit.'
    if not path.exists():
        args=['git','clone']
        if branch:
            args+=['--branch',branch,'--single-branch']
        subprocess.run([*args,url,str(path)],check=True)
    assert subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True).strip()==url
    assert not subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=no'],text=True).strip()
    exists=subprocess.run(['git','-C',str(path),'cat-file','-e',commit],capture_output=True)
    if exists.returncode:
        subprocess.run(['git','-C',str(path),'fetch','origin'],check=True)
    subprocess.run(['git','-C',str(path),'checkout','--detach',commit],check=True)

REPO=Path('/content/i-jepa-wm-prediction-replay')
DINO=Path('/content/dino_wm')
checkout('https://github.com/kmylonas/i-jepa-wm.git',REPO,runtime['source_commit'],'curriculum-cost-adapter')
checkout('https://github.com/gaoyuezhou/dino_wm.git',DINO,signature['dino_commit'])
subprocess.run([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml',
                'scipy','einops','pillow','imageio','imageio-ffmpeg'],check=True)
sys.path.insert(0,str(REPO))

from src.mylonas_ijepa_wm import plan_pointmaze as visual, plan_pointmaze_proprio as planner, plan_proprio
from src.mylonas_ijepa_wm.curriculum_runtime import PilotRuntime,file_sha256,source_digest
from src.mylonas_ijepa_wm.experience_curriculum import _raw_cost
from src.mylonas_ijepa_wm.environment_adapters import WallAdapter
from src.mylonas_ijepa_wm.environment_configs import EnvironmentConfig
from src.mylonas_ijepa_wm.plan_pointmaze import load_frozen_ijepa_encoder

assert source_digest(REPO/'src')==signature['source'], 'Encoder source differs from collection.'
assert source_digest(DINO/'env/wall')==signature['dino_source'], 'Renderer source differs from collection.'
goal_path=run/'collection_goals.pt'
assert file_sha256(goal_path)==stage['condition']['goal_sha256'], 'Saved collector goal tokens changed.'
targets=torch.load(goal_path,map_location='cpu',weights_only=True)
assert targets['runtime_id']==runtime['runtime_id'], 'Goal tensors belong to another runtime.'
target_tokens=targets['tokens']

device=torch.device('cuda')
variant=signature['runtime_variant']
torch.set_float32_matmul_precision(variant['matmul_precision'])
print('Collection GPU:',variant['device_name'],'· Torch:',signature['torch_version'])
print('Audit GPU:',torch.cuda.get_device_name(device),'· Torch:',str(torch.__version__))

def local_checkpoint(name,source,expected):
    path=Path('/content')/name
    if not path.is_file() or file_sha256(path)!=expected:
        shutil.copy2(source,path)
    assert file_sha256(path)==expected, f'Checksum differs for {source}'
    return path

drive_root=Path('/content/drive/MyDrive')
encoder_path=local_checkpoint('raw-mse-audit-encoder.pth.tar',
    drive_root/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar',signature['models']['encoder'])
wm_path=local_checkpoint('raw-mse-audit-world-model.pt',
    drive_root/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt',signature['models']['world_model'])
encoder=load_frozen_ijepa_encoder(encoder_path,device)
loader=SimpleNamespace(device=device,args=SimpleNamespace(world_model_checkpoint=wm_path),model=None)
model=PilotRuntime.load_world_model(loader).eval().requires_grad_(False)
metadata=loader.metadata
config=EnvironmentConfig(env='wall',wall_layouts=[case['layout']],dino_wm_root=str(DINO))
env=WallAdapter(config,planning['max_macro_steps']*config.actions_per_model_step)
precision=signature['precision']
target=target_tokens[case['goal_index']].to(device).float()
score=_raw_cost(target)

## Replay the commands in the world model and the simulator

The predicted rollout starts from the original frame and position, then feeds its own latent and proprioceptive predictions forward. The simulator replay starts from the same reset. Its continuation beyond the original stop is included only after verifying the recorded positions and pixels.

In [ ]:
def archived_plan(record,raw,stride,action_dim,horizon):
    macro=np.asarray(record['macro_actions'],dtype=np.float32)
    durations=record['macro_durations']
    primitive=raw['primitive_actions'].numpy()
    if (macro.shape!=(horizon,stride*action_dim) or len(durations)!=horizon
            or durations[:-1]!=[stride]*(horizon-1) or not 1<=durations[-1]<=stride
            or sum(durations)!=len(primitive) or not np.isfinite(macro).all()
            or not np.array_equal(macro.reshape(-1,action_dim)[:len(primitive)],primitive)):
        raise ValueError('The full commanded sequence is unavailable or its executed prefix differs.')
    return macro

def reset_case():
    return env.reset(case['seed'],layout=case['layout'],start=case['start'],goal=case['goal'])

stride=config.actions_per_model_step
horizon=planning['planning_horizon']
macro=archived_plan(record,raw,stride,config.micro_action_dim,horizon)
commands=macro.reshape(-1,config.micro_action_dim)
assert np.array_equal(reset_case().frame,raw['frames'][0].numpy()), 'Reset image differs.'
assert np.allclose(env.state()[:2],actual_xy[0],rtol=0,atol=1e-5), 'Reset position differs.'

if VERIFY_CEM:
    class FirstPlan(Exception):
        pass
    recovered=[]
    def observe_plan(step,plan,*_):
        assert step==0
        recovered.append(plan)
        raise FirstPlan
    initial=reset_case()
    try:
        plan_proprio.run_episode(env,config,encoder,model,metadata,SimpleNamespace(**planning,cost='latent'),
            device,precision,case['seed'],None,initial=initial,latent_cost_fn=score,plan_observer=observe_plan)
    except FirstPlan:
        pass
    recovered_plan=recovered[0]
    error=np.abs(recovered_plan.action_sequence.cpu().numpy()-macro)
    print('CEM reproduction · maximum absolute command error on recorded prefix:',float(error.reshape(-1,2)[:steps].max()))
    print('CEM reproduction · maximum absolute command error across all commands:',float(error.max()))
    print('CEM reproduced terminal MSE:',recovered_plan.cost)

initial_latent=visual.encode_frame(encoder,raw['frames'][0].numpy(),device,precision)
mean=metadata['state_mean'].to(device=device,dtype=torch.float32)
std=metadata['state_std'].to(device=device,dtype=torch.float32)
normalized=(raw['states'][0].to(device=device,dtype=torch.float32)-mean)/std
predicted_latents,predicted_states=planner.rollout_action_sequences(
    model,initial_latent[None,None].repeat(1,model.num_hist,1,1),
    normalized[None,None].repeat(1,model.num_hist,1),
    torch.zeros(model.num_hist-1,config.macro_action_dim,device=device),
    torch.from_numpy(macro)[None],device=device,chunk_size=1,
    action_mean=metadata.get('action_mean'),action_std=metadata.get('action_std'),precision=precision)
pred_xy=np.vstack([actual_xy[0],(predicted_states[0].float()*std+mean).cpu().numpy()[:,:2]])
pred_mse=torch.cat([score(initial_latent[None]),score(predicted_latents[0])]).cpu().numpy()
pred_steps=np.arange(horizon+1)*stride

def frame_cost(frame):
    tokens=visual.encode_frame(encoder,np.asarray(frame),device,precision)
    return float(score(tokens[None])[0])

true_mse=np.asarray([frame_cost(frame.numpy()) for frame in raw['frames']])
initial=reset_case()
replay_xy=[initial.state[:2]]
replay_frames=[initial.frame]
for command in commands:
    outcome=env.step(command)
    replay_xy.append(outcome.state[:2])
    replay_frames.append(outcome.frame)
replay_xy=np.asarray(replay_xy)
state_error=float(np.max(np.abs(replay_xy[:steps+1]-actual_xy)))
prefix_pixels=np.array_equal(np.stack(replay_frames[:steps+1]),raw['frames'].numpy())
verified=state_error<=1e-5 and prefix_pixels
print('Recorded terminal prediction scored by CEM:',record['final_planner_cost'])
print('Recomputed terminal prediction on archived sequence:',float(pred_mse[-1]))
print('Simulator prefix · max position error:',state_error,'· exact pixels:',prefix_pixels)
if not verified:
    print('The simulator continuation is excluded because the recorded prefix did not reproduce.')
tail_steps=np.arange(steps+1,len(commands)+1) if verified else np.array([],dtype=int)
tail_mse=np.asarray([frame_cost(replay_frames[i]) for i in tail_steps])

## Compare matched checkpoints

The position error is the Euclidean difference between the predicted and real ball positions after the same commands. Action 23 is recorded and marks the original success. Action 25 uses the verified diagnostic continuation. Predicted points are available only at macro boundaries, so the lines connecting them are a visual guide.

In [ ]:
goal=np.asarray(case['goal'])
distance=lambda xy:np.linalg.norm(xy-goal,axis=-1)
rows=[]
for j,step in enumerate(pred_steps):
    if step<=steps:
        xy,mse,source=actual_xy[step],true_mse[step],'recorded'
    elif verified:
        xy,mse,source=replay_xy[step],tail_mse[step-steps-1],'diagnostic continuation'
    else:
        continue
    rows.append(dict(action=step,real_source=source,predicted_x=pred_xy[j,0],predicted_y=pred_xy[j,1],
                     real_x=xy[0],real_y=xy[1],position_error=np.linalg.norm(pred_xy[j]-xy),
                     predicted_goal_distance=distance(pred_xy[j]),real_goal_distance=distance(xy),
                     predicted_mse=pred_mse[j],real_mse=mse))
table=pd.DataFrame(rows)
display(table.round(5))
table.to_csv('/content/raw_mse_prediction_replay.csv',index=False)

import matplotlib.pyplot as plt
from matplotlib.patches import Circle
fig=plt.figure(figsize=(13,8),constrained_layout=True)
layout=fig.add_gridspec(2,2)
path_ax=fig.add_subplot(layout[:,0])
distance_ax=fig.add_subplot(layout[0,1])
mse_ax=fig.add_subplot(layout[1,1],sharex=distance_ax)
path_ax.plot(*actual_xy.T,'o-',ms=3,color='tab:blue',label='Recorded')
path_ax.plot(*pred_xy.T,'s--',ms=5,color='tab:orange',label='Predicted macro states')
for j,step in enumerate(pred_steps[1:],1):
    path_ax.annotate(str(step),pred_xy[j],xytext=(4,4),textcoords='offset points',color='tab:orange')
path_ax.scatter(*goal,marker='*',color='black',s=120,label='Goal')
path_ax.add_patch(Circle(goal,SUCCESS_RADIUS,fill=False,ls=':',edgecolor='gray'))
path_ax.set(xlabel='Ball x',ylabel='Ball y',aspect='equal',title=CASE_ID)
path_ax.invert_yaxis()
for ax,real,pred,ylabel in [(distance_ax,distance(actual_xy),distance(pred_xy),'Distance to goal (units)'),
                          (mse_ax,true_mse,pred_mse,'Raw latent MSE')]:
    ax.plot(np.arange(steps+1),real,'o-',ms=3,color='tab:blue',label='Recorded')
    ax.plot(pred_steps,pred,'s--',ms=5,color='tab:orange',label='Predicted')
    ax.axvline(steps,color='gray',ls=':',label='Original stop')
    ax.set(ylabel=ylabel)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
if len(tail_steps):
    extra=np.r_[steps,tail_steps]
    path_ax.plot(*replay_xy[extra].T,'o:',ms=4,color='tab:green',label='Diagnostic continuation')
    distance_ax.plot(extra,distance(replay_xy[extra]),'o:',ms=4,color='tab:green',label='Diagnostic continuation')
    mse_ax.plot(extra,np.r_[true_mse[-1],tail_mse],'o:',ms=4,color='tab:green',label='Diagnostic continuation')
    distance_ax.legend(fontsize=8)
    mse_ax.legend(fontsize=8)
distance_ax.axhline(SUCCESS_RADIUS,color='gray',ls=':',alpha=.5)
path_ax.legend(fontsize=8)
mse_ax.set_xlabel('Executed action')
fig.savefig('/content/raw_mse_prediction_replay.png',dpi=150,bbox_inches='tight')
plt.show()